# EDERA — Walkthrough (all parameters exposed)

This notebook reproduces the pipeline **step by step** on a test dataset, calling the two public entry points — `compute_tx_topics` (Function 1) then `cluster_tx_topics` (Function 2) — and
rendering **every** diagnostic plot inline.

**Every exposed parameter is set explicitly in the config cells below** so you can change any of them and re-run. Each is annotated with its purpose; the value shown is either the library default or a small fixture-friendly override (marked `# fixture` in the comment).

In [ ]:
import os, glob
from IPython.display import Image, display
import pandas as pd

import EDERA
from EDERA import AdvancedConfig
print("EDERA", EDERA.__version__)

# Fixture path relative to this notebook (repo is installed as the EDERA pkg).
DATA = os.path.join("..", "data", "test_dataset.csv")
OUT = os.path.abspath("../results_smoke_test")
os.makedirs(OUT, exist_ok=True)

def show_pngs(pattern, width=560):
    for p in sorted(glob.glob(os.path.join(OUT, pattern))):
        print(os.path.basename(p))
        display(Image(filename=p, width=width))

## Parameters

Shared parameters are presented here: they are passed to **both** functions. `diagnostic_bbox` is in **raw input units** (the pipeline converts it to internal/scaled coordinates once). Set it to `None` to skip every spatial/edge plot.

In [ ]:
N_WORKERS       = 16                  # workers per parallel step; 1 = serial (reproducible), -1 = all cores
OUT_PATH        = OUT
PLOT_FORMAT     = ["png", "html"]     # list: pdf/png -> matplotlib, html -> plotly
DIAGNOSTIC_BBOX = (0, 100, 0, 100)    # raw units (xmin,xmax,ymin,ymax[,zmin,zmax]); None to skip
RANDOM_STATE    = 42
DPI             = 150                 # 600 for publication; 150 keeps this notebook light
SUPPRESS_WARN   = True                # silence third-party warning chatter

### `AdvancedConfig` — all internal parameters (likely you won't need to tune)
Every field is listed explicitly at its default. Set `dark_bg=True` for black spatial panels; set `umap_parallel=False` for seeded (reproducible, single-threaded) UMAPs.

In [ ]:
ADV = AdvancedConfig(
    # -- 2D auto-partitioning (used only when there is no `fov` column) --
    target_partition_size   = 100_000,  # approx transcripts per pseudo-FOV
    # -- parallelism --
    inner_threads           = 1,        # threads INSIDE each kNN worker (keep n_workers*inner<=cores)
    # -- HNSW clustering graph --
    ef_construction         = 200,      # HNSW build accuracy (higher=slower/more accurate)
    hnsw_M                  = 16,        # HNSW links per node
    transfer_k              = 15,        # neighbours for HNSW majority-vote label transfer
    # -- FSS consensus smoothing --
    consensus_threshold     = 0.8,      # neighbour-vote fraction needed to flip a label
    # -- LDA --
    lda_passes              = 10,       # gensim training passes
    lda_chunksize           = 2000,     # documents per LDA update
    lda_eta                 = "symmetric",  # per-topic gene prior
    # -- RWR diffusion --
    rwr_tolerance           = 1e-5,     # Frobenius convergence tolerance
    rwr_max_steps           = 1000,     # max RWR iterations
    # -- diagnostic UMAP determinism/speed (U7) --
    umap_parallel           = True,     # True=fast, multi-threaded, non-deterministic; False=seeded
    umap_n_neighbors        = 15,       # cluster-UMAP n_neighbors
    umap_min_dist           = 0.1,      # cluster-UMAP min_dist
    umap_subsample_fraction = 0.01,     # fraction of subsample used to FIT the cluster UMAP
    # -- diagnostic subsample caps (points rendered) --
    diag_hist_subsample         = 10_000,
    diag_jointplot_subsample    = 50_000,
    diag_scatter_subsample      = 100_000,
    cluster_umap_subsample      = 200_000,
    partition_overview_subsample = 200_000,
    # -- styling --
    dark_bg                 = False,    # black backgrounds for spatial scatters
)
ADV

### Function 1 — `compute_tx_topics` parameters
Load → RCB pseudo-FOVs → kNN → FSS → filter → METIS → stop-words → LDA → RWR.
Column names map the raw CSV to the canonical `x/y/z/target/fov`.

**Preset.** `F1_mode` picks one of two benchmarked settings: `"default"` (general-purpose; strength filter on, degree ≥ 5, 50 transcripts per pseudo-cell) or `"complex structures"` (touching cells, complex contiguous topologies; strength filter off, degree ≥ 2, 25 transcripts per pseudo-cell). `None` = the default values. The preset only fills `n_mad_strength`, `min_connected_fss` and `transcripts_per_cell` when you leave them at `None`; any value you set wins. Below, `transcripts_per_cell` is overridden for the small fixture.

In [ ]:
COMPUTE_PARAMS = dict(
    # -- source column names --
    x_name              = "x",
    y_name              = "y",
    z_name              = "z",
    gene_name           = "gene_id",    # fixture: gene label column
    fov_column          = "fov",        # absent in fixture -> RCB auto-partition
    # -- physical resolutions (raw -> microns) --
    x_res               = 1.0,          # fixture: already in microns
    y_res               = 1.0,          # fixture
    z_res               = 1.0,          # fixture
    # -- preset: "default" | "complex structures" | None (= default) --
    F1_mode             = "default",           # fills the three params below when they are None
    # -- graph construction --
    knn_neighbors       = 15,           # fixture (default 25)
    max_edge_length     = 4.0,          # absolute edge-length cap (internal units); None disables
    n_mad_strength      = None,         # MAD multiplier for node-strength cut; None = preset (4.0 default, 1e9 = off complex structures)
    min_connected_fss   = None,         # iterative low-degree prune; None = preset (5 / 2), 0 disables
    # -- METIS / LDA --
    transcripts_per_cell = 100,          # overrides the preset (50 default / 25 complex structures) for the fixture
    n_topics_lda        = 10,            # fixture (default 50); not part of the preset
    # -- stop-words --
    stopwords_thr       = "2MAD",       # doc-freq cutoff: "XMAD" = median+X*MAD, or a float in (0,1]; None disables
    stopword_min_docfreq = 0.02,        # only genes with doc_freq>=this enter the median/MAD stats
    # -- diffusion --
    damping_factor_rwr  = 0.98,         # RWR propagation weight c (restart = 1-c)
    # -- shared plumbing --
    n_workers           = N_WORKERS,
    out_path            = OUT_PATH,
    plot_format         = PLOT_FORMAT,
    diagnostic_bbox     = DIAGNOSTIC_BBOX,
    random_state        = RANDOM_STATE,
    dpi                 = DPI,
    suppress_warnings   = SUPPRESS_WARN,
    checkpoint_path     = None,         # set a dir to checkpoint Function 1 to disk
)


### Function 2 — `cluster_tx_topics` parameters
HNSW (+ FSS mix) → Leiden → merge small clusters → label transfer → FSS smoothing.

In [ ]:
CLUSTER_PARAMS = dict(
    sample_fraction_transfer_cluster = 0.5,   # (0,1]; 1.0 clusters ALL points, skips transfer
    n_neighbors_hnsw   = 15,            # fixture (default 100)
    leiden_resolution  = [0.1, 0.5],         # list -> one final_cluster_res<r> column each
    min_ratio_merge_small_clusters = 0.03,  # clusters below this fraction are merged
    smoothing_threshold = 0.01,         # stop smoothing when flipped-fraction < this
    min_delta          = 0.01,          # stop when successive fractions differ by < this
    max_loops          = 3,             # max smoothing loops per FOV
    fss_weight         = 0.5,           # FSS/HNSW convex mix (0=pure HNSW, 1=pure FSS)
    # -- shared plumbing --
    n_workers          = N_WORKERS,
    out_path           = OUT_PATH,
    plot_format        = PLOT_FORMAT,
    diagnostic_bbox    = DIAGNOSTIC_BBOX,
    random_state       = RANDOM_STATE,
    dpi                = DPI,
    suppress_warnings  = SUPPRESS_WARN,
)


## Function 1 — run

In [ ]:
result = EDERA.compute_tx_topics(data_path=DATA, advanced=ADV, **COMPUTE_PARAMS)

### Step 0 — RCB partition overview
No `fov` column -> 2D space split into balanced pseudo-FOVs.

In [ ]:
show_pngs("0a_partition_overview.png")

### Step 1 — raw FSS QC

In [ ]:
show_pngs("1a_fss_edge_weight_histogram.png")
show_pngs("1b_fss_edge_weight_vs_distance.png")

### Step 2 — unified filter + ROI FSS edges
A **single** FOV-independent ROI plot (`2c_fss_edges_bbox`) over all in-bbox
transcripts/edges: rocket-coloured edges, white nodes, black background.

In [ ]:
display(pd.read_csv(os.path.join(OUT, "2a_filter_stats.csv")))
show_pngs("2b_fss_post_filter_weight_vs_distance.png")
show_pngs("2c_fss_edges_bbox.png")

### Step 3 — METIS pseudo-cells
Three 3D viewpoints; z visually exaggerated 3x with true tick labels.

In [ ]:
show_pngs("3b_metis_spatial_view*.png")

### Step 4 — stop-word detection
Stop-words are flagged on their **document frequency** (fraction of METIS
pseudo-cells a gene appears in): `stopwords_thr="3MAD"` cuts at
`median(DF) + 3*MAD(DF)`, which on this fixture lands at DF ≈ 0.145 and flags
the ubiquitous tail (46/593 genes, with VIM / NEAT1 / MALAT1 far out at
DF ≈ 0.9). Raise the coefficient (`"5MAD"`) or pass a custom cutoff
(`stopwords_thr=0.5`) to be less aggressive. Genes are only `kept` or
`stopword` — there is no rare-gene removal. Red dashed line = the applied
threshold; grey dashed line = the doc-frequency cutoff of the stats pool.

In [ ]:
stats = pd.read_csv(os.path.join(OUT, "LDA_out", "4a_gene_stopword_stats.csv"), index_col=0)
display(stats[stats["reason"] == "stopword"])
show_pngs("LDA_out/4b_auto_stopwords.png")

### Step 5 — LDA gene embeddings

In [ ]:
show_pngs("LDA_out/5b_gene_embeddings_lda_umap.png")
display(pd.read_csv(os.path.join(OUT, "LDA_out", "5c_top_genes_per_topic.csv")))

### Step 6 — RWR diffusion (raw vs diffused topics)

In [ ]:
show_pngs("6a_topic_raw_vs_diffused_*.png")

## Function 2 — run

In [ ]:
df = EDERA.cluster_tx_topics(result=result, advanced=ADV, **CLUSTER_PARAMS)
display(pd.read_csv(os.path.join(OUT, "7a_cluster_counts.csv")))

### Step 7 — clustering diagnostics
Spatial cluster map (discrete legend), cluster UMAP, smoothing convergence.

In [ ]:
show_pngs("7d_cluster_spatial_*.png")
show_pngs("7e_cluster_umap_*.png")
show_pngs("7f_smoothing_convergence_*.png")